<a href="https://colab.research.google.com/github/kej534923-maker/card-1995-iv-replication/blob/main/01_Data_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files
uploaded = files.upload()


Saving nls.dat to nls.dat
Saving read1.sas to read1.sas


In [14]:
import os
print("read1.sas exists:", os.path.exists("read1.sas"))
print("read1.sas size:", os.path.getsize("read1.sas") if os.path.exists("read1.sas") else None)
with open("read1.sas", "r", errors="ignore") as f:
    for i in range(80):
        line = f.readline()
        if not line:
            break
        print(line.rstrip())

read1.sas exists: True
read1.sas size: 3076
comment program read1: reads nls data from flat file;
comment see codebook code_bk.txt;
data one;

title1 'input nls data set and run example regression';

infile '***put full address of file here***/nls.flt';

input id    /*sequential id*/
    nearc2  /*grew up near 2-yr college*/
    nearc4  /*4-yr college*/
    nearc4a /*4-yr public college*/
    nearc4b /*near 4-yr priv college*/
    ed76  /*educ in 1976*/
    ed66  /*educ in 1966*/
    age76 /*age in 1976*/
    daded /*dads education missing=avg*/
    nodaded /* 1 if dad ed imputed*/
    momed /*moms education*/
    nomomed /* 1 if mom ed imputed*/
    weight
    momdad14 /*1 if lived with mom and dad age 14*/
    sinmom14 /*1 if lived with single mom age 14*/
    step14  /*1 if lived with step parent age 14*/
    reg661 /* region=1 in 1966 */
    reg662
    reg663
    reg664
    reg665
    reg666
    reg667
    reg668
    reg669
    south66 /*lived in south in 1966*/
    work76 /* worke

In [15]:
import re

def extract_varlist_from_sas(path="read1.sas"):
    text = open(path, "r", errors="ignore").read()

    m = re.search(r"(?mi)^\s*input\s+(.*?);", text, flags=re.DOTALL)
    if not m:
        raise ValueError("Cannot find INPUT block (line-start input ... ;)")

    block = m.group(1)

    block = re.sub(r"/\*.*?\*/", " ", block, flags=re.DOTALL)

    tokens = re.findall(r"\b[A-Za-z_][A-Za-z0-9_]*\b", block)

    return tokens

colnames = extract_varlist_from_sas("read1.sas")
print("Number of variables:", len(colnames))
print("First 10:", colnames[:10])
print("Last 5:", colnames[-5:])

Number of variables: 52
First 10: ['id', 'nearc2', 'nearc4', 'nearc4a', 'nearc4b', 'ed76', 'ed66', 'age76', 'daded', 'nodaded']
Last 5: ['iq', 'marsta76', 'marsta78', 'marsta80', 'libcrd14']


In [16]:
import pandas as pd

df = pd.read_csv(
    "nls.dat",
    sep=r"\s+",
    header=None,
    names=colnames,
    engine="python"
)

print("df.shape:", df.shape)
df.head()

df.shape: (3613, 52)


,id,nearc2,nearc4,nearc4a,nearc4b,ed76,ed66,age76,daded,nodaded,...,noint80,enroll76,enroll78,enroll80,kww,iq,marsta76,marsta78,marsta80,libcrd14
0,2,0,0,0,0,7,5,29,9.94,1,...,0,0,0,0,15,.,1,1,1,0
1,3,0,0,0,0,12,11,27,8.00,0,...,0,0,0,0,35,93,1,4,4,1
2,4,0,0,0,0,12,12,34,14.00,0,...,1,0,.,.,42,103,1,.,.,1
3,5,1,1,1,0,11,11,27,11.00,0,...,0,0,.,0,25,88,1,.,5,1
4,6,1,1,1,0,12,12,34,8.00,0,...,1,0,0,.,34,108,1,1,.,0


In [12]:
import os
os.makedirs("data/processed", exist_ok=True)

In [13]:
df.to_parquet("data/processed/nls_clean.parquet", index=False)